## Creating Variations of an Image

In [ ]:
# Install required packages (run once; restart the kernel if needed)
%pip install openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load OPENAI_API_KEY from a .env file in the project directory.
# Create a .env file with the line: OPENAI_API_KEY=your-key-here
load_dotenv()

# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'  # hardcoded key (not recommended)

# Initialize the OpenAI client; it reads OPENAI_API_KEY from the environment
client = OpenAI()

# Note: images.create_variation() only supports 'dall-e-2' and 'gpt-image-1'.
# 'dall-e-3' does NOT support variations — it only supports images.generate().
# 'dall-e-2' returns a URL; 'gpt-image-1' returns base64-encoded image data.

# Open the source image in binary mode (the API requires raw bytes)
# Using 'with' ensures the file is properly closed after the request
with open('original_2.png', 'rb') as image:
    # Request a variation of the image from DALL-E
    # n: number of variations to generate; size: output resolution
    response = client.images.create_variation(
        model='dall-e-2',
        image=image,
        n=1,
        size='1024x1024'
    )

# The response contains a temporary URL pointing to the generated image
image_url = response.data[0].url
print(image_url)

## Edit an Existing Image

In [16]:
import os
import base64
from dotenv import load_dotenv
from openai import OpenAI

# Re-load environment variables (useful if running this section independently)
load_dotenv()

client = OpenAI()

# Note: images.edit() only supports 'dall-e-2' and 'gpt-image-1'.
# 'dall-e-3' does NOT support image editing — it only supports images.generate().
# 'dall-e-2' returns a URL; 'gpt-image-1' returns base64-encoded image data.

# Open the original image and its transparency mask in binary mode.
# The mask must be a PNG with an alpha channel:
# transparent (alpha=0) pixels mark the area DALL-E will inpaint.
with open('man_on_motorcycle.png', 'rb') as image, open('mask.png', 'rb') as mask:
    # Edit the image: DALL-E fills the masked region according to the prompt
    response = client.images.edit(
        model='gpt-image-1',
        image=image,
        mask=mask,
        prompt='John Wick riding a motorcycle on an apocalyptic field containing a big red moon in a cyberpunk look, high quality photography',
        n=1,
        size='1024x1024'
    )

# gpt-image-1 returns base64-encoded image data instead of a URL
image_data = base64.b64decode(response.data[0].b64_json)
output_path = 'edited_output.png'
with open(output_path, 'wb') as f:
    f.write(image_data)

print(f'Image saved to {output_path}')

Image saved to edited_output.png
